# Polynomial Regression — Model Complexity and Evaluation

**Course:** AIC 301 — Machine Learning  
**Instructor:** Dr. Kashif Sultan  
**Student:** Ahsan  
**Date:** October 14, 2026

---

## Problem Description

This notebook explores how **polynomial regression** can capture non-linear relationships that simple linear regression misses. We use the **Auto MPG dataset** (UCI Machine Learning Repository) to predict a car's fuel efficiency (`mpg`) from its engine power (`horsepower`).

The core question we investigate: *What polynomial degree best balances model complexity against generalization, and why is simply choosing the highest degree a bad strategy?*

### Dataset Summary
- **Source:** UCI Machine Learning Repository — Auto MPG (originally from Carnegie Mellon University)
- **Observations:** 398 car models (1970s–1980s era)
- **Target Variable:** `mpg` (miles per gallon) — continuous
- **Selected Input Feature:** `horsepower` — continuous
- **Why Regression?** The target `mpg` is a continuous numeric value, not a category, so this is a regression task.

In [ ]:
# ============================================================
# Step 1: Import Libraries
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: model building and evaluation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Plot settings for cleaner visuals
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

---
## Step 2: Load and Explore the Dataset

We load the Auto MPG dataset and perform basic exploration to understand its structure, check for missing values, and examine the relationship between `horsepower` and `mpg`.

In [ ]:
# ============================================================
# Step 2a: Load the dataset
# ============================================================

df = pd.read_csv('auto_mpg.csv')

# Quick look at the first 5 rows
print('First 5 rows of the dataset:')
df.head()

In [ ]:
# ============================================================
# Step 2b: Dataset shape, data types, and missing values
# ============================================================

print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
print()

print('Column data types:')
print(df.dtypes)
print()

print('Missing values per column:')
print(df.isnull().sum())
print()

print('Basic statistics:')
df.describe()

In [ ]:
# ============================================================
# Step 2c: Visualize the relationship — Horsepower vs MPG
# ============================================================

plt.figure(figsize=(10, 6))
plt.scatter(df['horsepower'], df['mpg'], alpha=0.5, edgecolors='k', linewidth=0.5)
plt.xlabel('Horsepower', fontsize=12)
plt.ylabel('Miles Per Gallon (MPG)', fontsize=12)
plt.title('Horsepower vs MPG — Is the Relationship Linear?', fontsize=14)
plt.tight_layout()
plt.savefig('plots/01_horsepower_vs_mpg_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print('Observation: The scatter plot shows a clear CURVED (non-linear) relationship.')
print('As horsepower increases, MPG decreases — but the drop is steep at first,')
print('then flattens out at higher horsepower values. A straight line would miss this curve.')

---
## Step 3: Data Cleaning and Train/Test Split

Before modeling, we handle missing values and split the data into training and test sets.

**Important:** We perform the split *before* fitting any transformations (like `PolynomialFeatures`). This prevents **data leakage** — the model should never see test data during training, not even indirectly through scaling or feature statistics.

In [ ]:
# ============================================================
# Step 3a: Handle missing values
# ============================================================

print(f'Rows before cleaning: {len(df)}')
print(f'Missing values in horsepower: {df["horsepower"].isnull().sum()}')

# Drop the 2 rows with missing horsepower values
# Reason: only 2 out of 398 rows (0.5%) — negligible data loss
# Alternative: we could fill with median, but dropping is simpler and more honest
df = df.dropna(subset=['horsepower'])

print(f'Rows after cleaning: {len(df)}')
print(f'Missing values remaining: {df.isnull().sum().sum()}')

In [ ]:
# ============================================================
# Step 3b: Select features and target
# ============================================================

# X = input feature (horsepower), y = target (mpg)
# We use double brackets [['horsepower']] to keep X as a 2D array (DataFrame)
# sklearn models require 2D input: (n_samples, n_features)
# Single brackets ['horsepower'] would give a 1D Series — sklearn would complain

X = df[['horsepower']]
y = df['mpg']

print(f'Feature matrix X shape: {X.shape}  — {X.shape[0]} samples, {X.shape[1]} feature')
print(f'Target vector y shape:  {y.shape}  — {y.shape[0]} values')

In [ ]:
# ============================================================
# Step 3c: Train/Test Split
# ============================================================

# 80% training, 20% testing
# random_state=42 ensures reproducibility — same split every time we run
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test set:     {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')
print()
print('The same train/test split will be used for ALL models (linear and polynomial)')
print('to ensure a fair comparison.')